In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()

print("Setup complete. DuckDB version:", duckdb.__version__)

Setup complete. DuckDB version: 1.5.0


In [2]:
con.execute("""
    CREATE TABLE members (
        member_id VARCHAR,
        date_of_birth DATE,
        gender VARCHAR
    )
""")

con.execute("""
    CREATE TABLE diagnoses (
        member_id VARCHAR,
        icd10_code VARCHAR,
        diagnosis_date DATE
    )
""")

con.execute("""
    CREATE TABLE encounters (
        member_id VARCHAR,
        encounter_date DATE,
        encounter_type VARCHAR
    )
""")

con.execute("""
    CREATE TABLE observations (
        member_id VARCHAR,
        observation_date DATE,
        measure_type VARCHAR,
        value_numeric FLOAT
    )
""")

print("Tables created successfully.")

Tables created successfully.


In [3]:
members_data = [
    ('M001', '1965-03-15', 'F'),  # Passes - BP controlled
    ('M002', '1970-07-22', 'M'),  # Fails - BP not controlled
    ('M003', '1958-11-30', 'F'),  # Excluded - ESRD diagnosis
    ('M004', '1980-04-10', 'M'),  # Fails - no BP reading on file
    ('M005', '1945-09-05', 'F'),  # Excluded - hospice
    ('M006', '1972-06-18', 'M'),  # Passes - BP controlled
    ('M007', '1963-12-01', 'F'),  # Fails - BP not controlled
    ('M008', '1955-08-25', 'M'),  # Passes - COVID impact flagged
    ('M009', '1968-02-14', 'F'),  # Passes - BP controlled
    ('M010', '1990-05-30', 'M'),  # Fails - no hypertension diagnosis
]

con.executemany("INSERT INTO members VALUES (?, ?, ?)", members_data)
print(f"Loaded {len(members_data)} members.")

Loaded 10 members.


In [4]:
diagnoses_data = [
    ('M001', 'I10',   '2024-01-15'),  # Hypertension
    ('M002', 'I10',   '2024-02-20'),  # Hypertension
    ('M003', 'I10',   '2024-01-10'),  # Hypertension
    ('M003', 'N18.6', '2024-01-10'),  # ESRD - will be excluded
    ('M004', 'I10',   '2024-03-05'),  # Hypertension
    ('M005', 'I10',   '2024-01-20'),  # Hypertension
    ('M005', 'Z51.5', '2024-01-20'),  # Hospice - will be excluded
    ('M006', 'I10',   '2024-02-10'),  # Hypertension
    ('M007', 'I10',   '2024-04-15'),  # Hypertension
    ('M008', 'I10',   '2024-01-05'),  # Hypertension
    ('M008', 'U07.1', '2024-03-10'),  # COVID - will be flagged
    ('M009', 'I10',   '2024-02-28'),  # Hypertension
    # M010 intentionally has no hypertension diagnosis
]

con.executemany("INSERT INTO diagnoses VALUES (?, ?, ?)", diagnoses_data)
print(f"Loaded {len(diagnoses_data)} diagnosis records.")

Loaded 12 diagnosis records.


In [5]:
encounters_data = [
    ('M001', '2024-02-10', 'outpatient'),
    ('M002', '2024-03-15', 'outpatient'),
    ('M003', '2024-01-20', 'outpatient'),
    ('M004', '2024-04-05', 'outpatient'),
    ('M005', '2024-01-25', 'hospice'),    # Hospice encounter type
    ('M006', '2024-02-20', 'outpatient'),
    ('M007', '2024-05-10', 'outpatient'),
    ('M008', '2024-02-15', 'outpatient'),
    ('M009', '2024-03-01', 'outpatient'),
    ('M010', '2024-06-15', 'outpatient'), # Has visit but no hypertension dx
]

con.executemany("INSERT INTO encounters VALUES (?, ?, ?)", encounters_data)
print(f"Loaded {len(encounters_data)} encounter records.")

Loaded 10 encounter records.


In [6]:
observations_data = [
    ('M001', '2024-02-10', 'BP_systolic',  128.0),
    ('M001', '2024-02-10', 'BP_diastolic',  82.0),
    ('M002', '2024-03-15', 'BP_systolic',  155.0),
    ('M002', '2024-03-15', 'BP_diastolic',  95.0),
    ('M003', '2024-01-20', 'BP_systolic',  130.0),
    ('M003', '2024-01-20', 'BP_diastolic',  84.0),
    ('M005', '2024-01-25', 'BP_systolic',  118.0),
    ('M005', '2024-01-25', 'BP_diastolic',  76.0),
    ('M006', '2024-02-20', 'BP_systolic',  134.0),
    ('M006', '2024-02-20', 'BP_diastolic',  86.0),
    ('M007', '2024-05-10', 'BP_systolic',  162.0),
    ('M007', '2024-05-10', 'BP_diastolic',  98.0),
    ('M008', '2024-02-15', 'BP_systolic',  136.0),
    ('M008', '2024-02-15', 'BP_diastolic',  88.0),
    ('M009', '2024-03-01', 'BP_systolic',  122.0),
    ('M009', '2024-03-01', 'BP_diastolic',  78.0),
]

con.executemany("INSERT INTO observations VALUES (?, ?, ?, ?)", observations_data)
print(f"Loaded {len(observations_data)} observation records.")

Loaded 16 observation records.


In [7]:
denominator = con.execute("""
    SELECT DISTINCT
        m.member_id,
        m.date_of_birth,
        m.gender,
        DATEDIFF('year', m.date_of_birth, DATE '2024-12-31') AS age
    FROM members m
    INNER JOIN diagnoses d
        ON m.member_id = d.member_id
        AND d.icd10_code = 'I10'
    INNER JOIN encounters e
        ON m.member_id = e.member_id
        AND e.encounter_date BETWEEN DATE '2024-01-01' AND DATE '2024-12-31'
        AND e.encounter_type = 'outpatient'
    WHERE DATEDIFF('year', m.date_of_birth, DATE '2024-12-31') BETWEEN 18 AND 85
    AND m.member_id NOT IN (
        SELECT member_id FROM diagnoses WHERE icd10_code = 'N18.6'
    )
    AND m.member_id NOT IN (
        SELECT member_id FROM diagnoses WHERE icd10_code = 'Z51.5'
    )
""").df()

print(f"Denominator count: {len(denominator)}")
print(denominator)

Denominator count: 7
  member_id date_of_birth gender  age
0      M006    1972-06-18      M   52
1      M009    1968-02-14      F   56
2      M001    1965-03-15      F   59
3      M007    1963-12-01      F   61
4      M008    1955-08-25      M   69
5      M004    1980-04-10      M   44
6      M002    1970-07-22      M   54


In [8]:
numerator = con.execute("""
    SELECT
        o_sys.member_id,
        o_sys.observation_date,
        o_sys.value_numeric AS systolic,
        o_dia.value_numeric AS diastolic,
        CASE
            WHEN o_sys.value_numeric < 140
            AND o_dia.value_numeric < 90
            THEN 'Controlled'
            ELSE 'Not Controlled'
        END AS bp_status
    FROM observations o_sys
    INNER JOIN observations o_dia
        ON o_sys.member_id = o_dia.member_id
        AND o_sys.observation_date = o_dia.observation_date
        AND o_dia.measure_type = 'BP_diastolic'
    WHERE o_sys.measure_type = 'BP_systolic'
    AND o_sys.member_id IN (
        SELECT member_id FROM denominator
    )
""").df()

print(f"Members with BP readings: {len(numerator)}")
print(numerator)

Members with BP readings: 6
  member_id observation_date  systolic  diastolic       bp_status
0      M001       2024-02-10     128.0       82.0      Controlled
1      M002       2024-03-15     155.0       95.0  Not Controlled
2      M006       2024-02-20     134.0       86.0      Controlled
3      M007       2024-05-10     162.0       98.0  Not Controlled
4      M008       2024-02-15     136.0       88.0      Controlled
5      M009       2024-03-01     122.0       78.0      Controlled


In [9]:
covid_analysis = con.execute("""
    SELECT
        n.member_id,
        n.observation_date AS bp_reading_date,
        n.systolic,
        n.diastolic,
        n.bp_status,
        d.diagnosis_date AS covid_diagnosis_date,
        DATEDIFF('day', d.diagnosis_date, n.observation_date) AS days_between_covid_and_bp,
        CASE
            WHEN d.member_id IS NOT NULL
            AND ABS(DATEDIFF('day', d.diagnosis_date, n.observation_date)) <= 90
            THEN 'Yes'
            ELSE 'No'
        END AS covid_impact_flag
    FROM numerator n
    LEFT JOIN diagnoses d
        ON n.member_id = d.member_id
        AND d.icd10_code = 'U07.1'
""").df()

print("CBP Measure Results with COVID Impact Flag:")
print(covid_analysis)

CBP Measure Results with COVID Impact Flag:
  member_id bp_reading_date  systolic  diastolic       bp_status  \
0      M008      2024-02-15     136.0       88.0      Controlled   
1      M001      2024-02-10     128.0       82.0      Controlled   
2      M002      2024-03-15     155.0       95.0  Not Controlled   
3      M006      2024-02-20     134.0       86.0      Controlled   
4      M007      2024-05-10     162.0       98.0  Not Controlled   
5      M009      2024-03-01     122.0       78.0      Controlled   

  covid_diagnosis_date  days_between_covid_and_bp covid_impact_flag  
0           2024-03-10                        -24               Yes  
1                  NaT                       <NA>                No  
2                  NaT                       <NA>                No  
3                  NaT                       <NA>                No  
4                  NaT                       <NA>                No  
5                  NaT                       <NA>          

In [10]:
final_report = con.execute("""
    SELECT
        m.member_id,
        m.gender,
        DATEDIFF('year', m.date_of_birth, DATE '2024-12-31') AS age,
        CASE
            WHEN n.member_id IS NULL THEN 'Missing - No BP Reading'
            ELSE n.bp_status
        END AS bp_status,
        CASE
            WHEN n.member_id IS NULL THEN NULL
            ELSE n.systolic
        END AS systolic,
        CASE
            WHEN n.member_id IS NULL THEN NULL
            ELSE n.diastolic
        END AS diastolic,
        CASE
            WHEN c.covid_impact_flag = 'Yes' THEN 'Yes'
            ELSE 'No'
        END AS covid_impact_flag,
        CASE
            WHEN n.member_id IS NULL THEN 'Fail - Missing Data'
            WHEN n.bp_status = 'Controlled' THEN 'Pass'
            ELSE 'Fail - Uncontrolled BP'
        END AS measure_result
    FROM denominator m
    LEFT JOIN numerator n
        ON m.member_id = n.member_id
    LEFT JOIN covid_analysis c
        ON m.member_id = c.member_id
""").df()

total = len(final_report)
passed = len(final_report[final_report['measure_result'] == 'Pass'])
rate = round(passed / total * 100, 1)

print("=" * 60)
print("HEDIS CBP MEASURE REPORT - Measurement Year 2024")
print("=" * 60)
print(final_report.to_string(index=False))
print("=" * 60)
print(f"Denominator:        {total}")
print(f"Numerator (passed): {passed}")
print(f"Measure Rate:       {rate}%")
print("=" * 60)

HEDIS CBP MEASURE REPORT - Measurement Year 2024
member_id gender  age               bp_status  systolic  diastolic covid_impact_flag         measure_result
     M006      M   52              Controlled     134.0       86.0                No                   Pass
     M009      F   56              Controlled     122.0       78.0                No                   Pass
     M001      F   59              Controlled     128.0       82.0                No                   Pass
     M007      F   61          Not Controlled     162.0       98.0                No Fail - Uncontrolled BP
     M008      M   69              Controlled     136.0       88.0               Yes                   Pass
     M002      M   54          Not Controlled     155.0       95.0                No Fail - Uncontrolled BP
     M004      M   44 Missing - No BP Reading       NaN        NaN                No    Fail - Missing Data
Denominator:        7
Numerator (passed): 4
Measure Rate:       57.1%
